# A/B Test Conversion Analysis

Statistical analysis of A/B test results using two-proportion z-test.

**Input:** `data/prepared_data.csv` — session-level data with test group assignments and conversion flags  
**Output:** `data/ab_testing_results_corrected.csv` — metrics and statistical significance per test and metric


In [ ]:
from statsmodels.stats.proportion import proportions_ztest
import pandas as pd
import numpy as np

In [ ]:
# Load data
df = pd.read_csv('../data/prepared_data.csv')

print('First rows:')
print(df.head())

print('\nTable info:')
print(df.info())

print('\nData validation:')
print(f'Total rows: {len(df)}')
print(f'Unique sessions: {df["ga_session_id"].nunique()}')

duplicate_sessions = df[df.duplicated(subset=['ga_session_id'], keep=False)]
if len(duplicate_sessions) > 0:
    print(f'Duplicate sessions: {duplicate_sessions["ga_session_id"].nunique()}')
else:
    print('No duplicate sessions found')

print('\nSession distribution by test and group:')
print(df.groupby(['test', 'test_group'])['ga_session_id'].nunique())

In [ ]:
# Metrics definition
metrics = {
    'add_payment_info / session': ('has_add_payment_info', 'session'),
    'add_shipping_info / session': ('has_add_shipping_info', 'session'),
    'begin_checkout / session': ('has_begin_checkout', 'session'),
    'new_accounts / session': ('has_new_account', 'session')
}

results = []

# Calculate metrics per test
for test_number in sorted(df['test'].dropna().unique()):
    test_data = df[df['test'] == test_number]
    control_data = test_data[test_data['test_group'] == 1]
    test_group_data = test_data[test_data['test_group'] == 2]

    for metric_name, (metric_column, denominator_type) in metrics.items():

        if denominator_type == 'session':
            denominator_count_control = control_data['ga_session_id'].nunique()
            denominator_count_test = test_group_data['ga_session_id'].nunique()
        else:
            denominator_count_control = np.nan
            denominator_count_test = np.nan

        numerator_count_control = control_data.loc[
            control_data[metric_column] == 1, 'ga_session_id'
        ].nunique()

        numerator_count_test = test_group_data.loc[
            test_group_data[metric_column] == 1, 'ga_session_id'
        ].nunique()

        conversion_rate_control = (
            numerator_count_control / denominator_count_control
            if denominator_count_control > 0 else np.nan
        )

        conversion_rate_test = (
            numerator_count_test / denominator_count_test
            if denominator_count_test > 0 else np.nan
        )

        if pd.notna(conversion_rate_control) and conversion_rate_control != 0:
            metric_change = (
                (conversion_rate_test - conversion_rate_control)
                / conversion_rate_control
            ) * 100
        else:
            metric_change = np.nan

        if denominator_count_control > 0 and denominator_count_test > 0:
            z_stat, p_value = proportions_ztest(
                count=[numerator_count_test, numerator_count_control],
                nobs=[denominator_count_test, denominator_count_control],
                alternative='two-sided'
            )
        else:
            z_stat, p_value = np.nan, np.nan

        results.append({
            'test_number': int(test_number),
            'metric': metric_name,
            'numerator_count_test': int(numerator_count_test),
            'denominator_count_test': int(denominator_count_test),
            'conversion_rate_test': round(conversion_rate_test, 6) if pd.notna(conversion_rate_test) else np.nan,
            'numerator_count_control': int(numerator_count_control),
            'denominator_count_control': int(denominator_count_control),
            'conversion_rate_control': round(conversion_rate_control, 6) if pd.notna(conversion_rate_control) else np.nan,
            'metric_change_%': round(metric_change, 2) if pd.notna(metric_change) else np.nan,
            'z_stat': round(z_stat, 4) if pd.notna(z_stat) else np.nan,
            'p_value': round(p_value, 6) if pd.notna(p_value) else np.nan,
            'significant': bool(p_value < 0.05) if pd.notna(p_value) else False
        })

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv('../data/ab_testing_results_corrected.csv', index=False)

print('\nResults:')
print(results_df)

print('\nStatistical significance:')
print(results_df['significant'].value_counts(dropna=False))